<a href="https://colab.research.google.com/github/AarnavSawant/KVCompass/blob/main/KVCompass.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# KVCompass Team Demo Notebook

This notebook is organized around **5 diverse methods** and **5 scenarios** so the team can split work cleanly before Friday. Each teammate should run only their assigned section, then one person can use the final aggregation cells to present the combined results.


## Shared Experiment Plan

**Methods used across the demo**
- `no_compression`
- `snapkv`
- `expected_attention`
- `adakv_expected`
- `streaming_llm`

**Five scenarios**
- Retrieval Long
- Reasoning Long
- Context Sweep
- Prefix Serving
- Memory Tight

**Team split**
- Tony: Retrieval Long
- Will: Reasoning Long
- Grady: Context Sweep
- Jamez: Prefix Serving
- Aarnav: Memory Tight + final aggregation


In [ ]:
# Shared setup: clone/refresh the repo, switch to the demo branch, and install dependencies.
from pathlib import Path

BRANCH_NAME = 'codex/colab-sweep-workflow'
repo_dir = Path('/content/KVCompass')
if not repo_dir.exists():
    !git clone https://github.com/AarnavSawant/KVCompass.git /content/KVCompass

%cd /content/KVCompass
!git fetch origin
!git checkout "$BRANCH_NAME"
!git pull origin "$BRANCH_NAME"
!nvidia-smi
!python -m pip install --upgrade pip
!pip install -r requirements.txt
!pip install -e .


In [ ]:
# Shared auth: set HF_TOKEN from Colab secrets and verify the login.
from google.colab import userdata
from huggingface_hub import whoami
import os

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
print(whoami())


In [ ]:
# Shared values used by all run cells.
MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'
METHODS = ['no_compression', 'snapkv', 'expected_attention', 'adakv_expected', 'streaming_llm']
TORCH_DTYPE = 'bfloat16'
SMOKE_MAX_CASES = None


## Tony: Retrieval Long

Run the next cell only. This covers the synthetic `retrieval_long` scenario across the five shared methods.


In [ ]:
# Run cell for Assignment 1.
!python scripts/run_kvpress_eval.py \
  --model "$MODEL_NAME" \
  --scenario retrieval_long \
  --method no_compression \
  --method snapkv \
  --method expected_attention \
  --method adakv_expected \
  --method streaming_llm \
  --device auto \
  --torch-dtype "$TORCH_DTYPE" \
  --output results/raw/assignment_1.csv \
  --run-name assignment_1


## Will: Reasoning Long

Run the next cell only. This covers the synthetic `reasoning_long` scenario across the five shared methods.


In [ ]:
# Run cell for Assignment 2.
!python scripts/run_kvpress_eval.py \
  --model "$MODEL_NAME" \
  --scenario reasoning_long \
  --method no_compression \
  --method snapkv \
  --method expected_attention \
  --method adakv_expected \
  --method streaming_llm \
  --device auto \
  --torch-dtype "$TORCH_DTYPE" \
  --output results/raw/assignment_2.csv \
  --run-name assignment_2


## Grady: Context Sweep

Run the next cell only. This covers the synthetic `context_sweep` scenario across the five shared methods.


In [ ]:
# Run cell for Assignment 3.
!python scripts/run_kvpress_eval.py \
  --model "$MODEL_NAME" \
  --scenario context_sweep \
  --method no_compression \
  --method snapkv \
  --method expected_attention \
  --method adakv_expected \
  --method streaming_llm \
  --device auto \
  --torch-dtype "$TORCH_DTYPE" \
  --output results/raw/assignment_3.csv \
  --run-name assignment_3


## Jamez: Prefix Serving

Run the next cell only. This covers the synthetic `prefix_serving` scenario across the five shared methods.


In [ ]:
# Run cell for Assignment 4.
!python scripts/run_kvpress_eval.py \
  --model "$MODEL_NAME" \
  --scenario prefix_serving \
  --method no_compression \
  --method snapkv \
  --method expected_attention \
  --method adakv_expected \
  --method streaming_llm \
  --device auto \
  --torch-dtype "$TORCH_DTYPE" \
  --output results/raw/assignment_4.csv \
  --run-name assignment_4


## Aarnav: Memory Tight

Run the next cell only. This covers the synthetic `memory_tight` scenario across the five shared methods.


In [ ]:
# Run cell for Assignment 5.
!python scripts/run_kvpress_eval.py \
  --model "$MODEL_NAME" \
  --scenario memory_tight \
  --method no_compression \
  --method snapkv \
  --method expected_attention \
  --method adakv_expected \
  --method streaming_llm \
  --device auto \
  --torch-dtype "$TORCH_DTYPE" \
  --output results/raw/assignment_5.csv \
  --run-name assignment_5


## Final Aggregation For The Presentation

After everyone shares their outputs, run the next cells in one Colab session or locally from the synced repo.


In [ ]:
# Combine the scenario CSVs.
from pathlib import Path
import pandas as pd

raw_paths = [Path(f"results/raw/assignment_{i}.csv") for i in range(1, 6) if Path(f"results/raw/assignment_{i}.csv").exists()]
raw_df = pd.concat([pd.read_csv(path) for path in raw_paths], ignore_index=True) if raw_paths else pd.DataFrame()
print("Scenario files:", [str(p) for p in raw_paths])
display(raw_df.head())

if not raw_df.empty:
    synthetic_summary = raw_df.groupby(["scenario", "method", "budget", "context_length"], dropna=False)[["latency_seconds", "throughput_tokens_per_second", "peak_gpu_memory_mb", "quality_score"]].mean(numeric_only=True).reset_index()
    display(synthetic_summary)


In [ ]:
# Build a compact leaderboard across all five scenarios.
import pandas as pd

if "synthetic_summary" in globals() and not synthetic_summary.empty:
    leaderboard = synthetic_summary.groupby(["method", "budget"], dropna=False)[["quality_score", "latency_seconds", "throughput_tokens_per_second", "peak_gpu_memory_mb"]].mean(numeric_only=True).reset_index()
    leaderboard = leaderboard.rename(columns={
        "quality_score": "avg_quality",
        "latency_seconds": "avg_latency_seconds",
        "throughput_tokens_per_second": "avg_throughput_tokens_per_second",
    }).sort_values(["avg_quality", "avg_latency_seconds"], ascending=[False, True])
    display(leaderboard)
else:
    print("No scenario summary found yet.")


In [ ]:
# Presentation plots from the combined leaderboard.
import matplotlib.pyplot as plt

if "leaderboard" in globals() and not leaderboard.empty:
    plot_df = leaderboard.copy()
    plot_df["label"] = plot_df["method"] + " @ " + plot_df["budget"].astype(str)
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    axes[0].bar(plot_df["label"], plot_df["avg_quality"], color="#2a6f97")
    axes[0].set_title("Average Quality")
    axes[0].tick_params(axis="x", rotation=60)
    axes[1].bar(plot_df["label"], plot_df["avg_latency_seconds"], color="#c1121f")
    axes[1].set_title("Average Latency (s)")
    axes[1].tick_params(axis="x", rotation=60)
    axes[2].bar(plot_df["label"], plot_df["peak_gpu_memory_mb"], color="#6a994e")
    axes[2].set_title("Peak GPU Memory (MB)")
    axes[2].tick_params(axis="x", rotation=60)
    plt.tight_layout()
    plt.show()
else:
    print("Leaderboard not ready yet.")


In [ ]:
# Simple recommendation table for the presentation.
import pandas as pd

if "leaderboard" in globals() and not leaderboard.empty:
    best_quality = leaderboard.sort_values(["avg_quality", "avg_latency_seconds"], ascending=[False, True]).iloc[0]
    best_latency = leaderboard.sort_values("avg_latency_seconds", ascending=True).iloc[0]
    best_memory = leaderboard.dropna(subset=["peak_gpu_memory_mb"]).sort_values("peak_gpu_memory_mb", ascending=True).iloc[0] if leaderboard["peak_gpu_memory_mb"].notna().any() else None
    rec_rows = [
        {"category": "best_for_quality", "method": best_quality["method"], "budget": best_quality["budget"]},
        {"category": "best_for_latency", "method": best_latency["method"], "budget": best_latency["budget"]},
    ]
    if best_memory is not None:
        rec_rows.append({"category": "best_for_memory", "method": best_memory["method"], "budget": best_memory["budget"]})
    display(pd.DataFrame(rec_rows))
else:
    print("Leaderboard not ready yet.")


In [ ]:
# Optional smoke test: one tiny scenario run to make sure the environment works.
!python scripts/run_kvpress_eval.py \
  --model Qwen/Qwen2.5-1.5B-Instruct \
  --scenario retrieval_long \
  --method no_compression \
  --device auto \
  --torch-dtype bfloat16 \
  --output results/raw/smoke_test.csv \
  --run-name smoke_test
